# ==========================================================
# 06_answer_cold_run.ipynb - Executive Visuals & Board Export
# ==========================================================

In [ ]:
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment

In [ ]:
PROJECT_ROOT = pathlib.Path.cwd().parent
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"
SLIDES_DIR = PROJECT_ROOT / "slides"
SLIDES_DIR.mkdir(exist_ok=True)

df_bookings = pd.read_csv(CLEAN_DIR / "bookings_clean.csv.gz", parse_dates=["slot_at"])

In [ ]:
# Define Half-Year periods
df_bookings["half"] = np.where(
    df_bookings["slot_at"].between("2025-01-01", "2025-06-30 23:59:59"), "H1_2025",
    np.where(df_bookings["slot_at"].between("2026-01-01", "2026-06-30 23:59:59"), "H1_2026", "other")
)

In [ ]:
# ----------------------------------------------------------
# Step 1: Headline Growth Calculation
# ----------------------------------------------------------
h1_2025_total = (df_bookings["half"] == "H1_2025").sum()
h1_2026_total = (df_bookings["half"] == "H1_2026").sum()
network_growth_pct = ((h1_2026_total / h1_2025_total) - 1) * 100

print(f"Network H1 2025 Tests: {h1_2025_total:,}")
print(f"Network H1 2026 Tests: {h1_2026_total:,}")
print(f"Headline Growth Percentage: {network_growth_pct:+.2f}%")

In [ ]:
# ----------------------------------------------------------
# Step 2: Render & Save Board Growth Chart
# ----------------------------------------------------------
df_city_summary = df_bookings[df_bookings["half"] != "other"].groupby(["city", "half"]).size().unstack("half")
df_city_summary["growth_pct"] = ((df_city_summary["H1_2026"] / df_city_summary["H1_2025"]) - 1) * 100

fig, ax = plt.subplots(figsize=(8, 4))
df_city_summary["growth_pct"].sort_values().plot(kind="barh", color="#4E38AB", ax=ax)
ax.set_title(f"H1 Test Growth by City (Overall Network: {network_growth_pct:+.1f}%)")
ax.set_xlabel("Growth %")
plt.tight_layout()

chart_path = SLIDES_DIR / "slide4_chart.png"
fig.savefig(chart_path, dpi=200)
plt.close(fig)
print(f"Saved chart graphic to: {chart_path}")

In [ ]:
# ----------------------------------------------------------
# Step 3: Export Excel Board Summary File
# ----------------------------------------------------------
excel_workbook = Workbook()
active_sheet = excel_workbook.active
active_sheet.title = "Board Number Summary"

summary_rows = [
    ("Metric Name", "Value"),
    ("Headline Growth (%)", f"{network_growth_pct:+.1f}%"),
    ("H1 2025 Actual Volume", int(h1_2025_total)),
    ("H1 2026 Actual Volume", int(h1_2026_total)),
    ("Target Plan Growth (%)", "+18.0%"),
    ("Primary Weakness", "Visits per person drop"),
]

for row_data in summary_rows:
    active_sheet.append(list(row_data))

active_sheet["A1"].font = Font(bold=True, size=12)
active_sheet["B1"].font = Font(bold=True, size=12)

excel_path = SLIDES_DIR / "board_number.xlsx"
excel_workbook.save(excel_path)
print(f"Saved Excel board report to: {excel_path}")